In [ ]:
# --- repo-root guard ---
# Notebooks live in notebooks/, but all data/code paths are relative to the repo root.
# If launched with the working directory set to notebooks/, step up one level so that
# 'src/', 'data/', 'pride_data/' and 'figures/' resolve correctly. Idempotent / no-op at root.
import os, pathlib
_cwd = pathlib.Path.cwd()
if not (_cwd / 'src').exists() and (_cwd.parent / 'src').exists():
    os.chdir(_cwd.parent)
print('working directory:', pathlib.Path.cwd())


In [27]:
# --- output directories (created relative to repo root) ---
from pathlib import Path as _P
for _d in ('figures','figures/figure2','figures/figure3','figures/figure4','figures/figure5'):
    _P(_d).mkdir(parents=True, exist_ok=True)

import sys, os
sys.path.insert(1, os.path.abspath(os.path.join("src", "alphaquant")))
#import alphaquant.run_pipeline as aq_pipeline

In [28]:
import sys, os
sys.path.insert(0, os.path.abspath("src"))
from alphaPhosHelperFunctions import *
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from scipy import stats
import plotly.figure_factory as ff
import kinase_library as kl
import analytics_core_V04 as ac
import matplotlib.pyplot as plt
from core import *
from matplotlib_venn import venn3

In [29]:
color_palette = ['#EEA69B', '#E78373', '#E1604C', '#DB452E', '#B3321E', '#8C2718', '#641C11']

In [30]:
import re
from pathlib import Path
import pandas as pd

RAW_DIR = Path('pride_data/analysis_data/revision/figure2')
INPUT_NG_ORDER = [10, 20, 50, 100, 200, 500, 1000]

def parse_filename(fname):
    """Return (workflow, condition, input_ng) from a Spectronaut report filename."""
    fname = fname.strip()
    workflow  = 'nanophos' if 'nanoPhos' in fname else ('uphos' if 'uPhos' in fname else 'unknown')
    if 'withEGF' in fname:
        condition = 'withegf'
    elif 'noEGF' in fname or 'woEGF' in fname:
        condition = 'noegf'
    elif 'HeLa' in fname:
        condition = 'hela'
    else:
        condition = 'unknown'
    m = re.search(r'(\d+)\s*ng', fname)
    return workflow, condition, int(m.group(1)) if m else None

buckets = {}
for path in sorted(RAW_DIR.glob('*.tsv')):
    w, c, n = parse_filename(path.name)
    if n is None:
        print(f"  ! skipping (no input found): {path.name}")
        continue
    buckets.setdefault(f"{w}_{c}", {})[n] = path

loaded = {}
for group, files in buckets.items():
    print(f"\n=== {group} ===")
    loaded[group] = {}
    for ng in sorted(files):
        df = pd.read_csv(files[ng], sep='\t')
        loaded[group][ng] = df
        print(f"  {ng:>5} ng: {len(df):>8,} rows  ({files[ng].name})")

nanophos_noEGF   = loaded.get('nanophos_noegf',   {})
uphos_hela       = loaded.get('uphos_hela',       {})
nanophos_withEGF = loaded.get('nanophos_withegf', {})


l_nanophos_noEGF   = [nanophos_noEGF[n]   for n in INPUT_NG_ORDER if n in nanophos_noEGF]
l_uphos_hela       = [uphos_hela[n]       for n in INPUT_NG_ORDER if n in uphos_hela]
l_nanophos_withEGF = [nanophos_withEGF[n] for n in INPUT_NG_ORDER if n in nanophos_withEGF]

  ! skipping (no input found): 20260518_120705_nanoPhos_dilser_withEGF_repeat_all_Report.tsv
  ! skipping (no input found): 20260518_162116_nanoPhos_optimization_salt_Report.tsv
  ! skipping (no input found): 20260622_091357_nanoPhos_dilser_withEGF_repeat_all_wo_norm_Report.tsv

=== nanophos_noegf ===
     10 ng:    5,461 rows  (20260506_132422_nanoPhos_dilser_woEGF_10ng_Report.tsv)
     20 ng:    6,756 rows  (20260506_132710_nanoPhos_dilser_woEGF_20ng_Report.tsv)
     50 ng:    8,282 rows  (20260506_140502_nanoPhos_dilser_woEGF_50ng_Report.tsv)
    100 ng:   18,260 rows  (20260506_140541_nanoPhos_dilser_noEGF_100ng_Report.tsv)
    200 ng:   18,847 rows  (20260506_140735_nanoPhos_dilser_noEGF_200ng_Report.tsv)
    500 ng:   27,833 rows  (20260506_141115_nanoPhos_dilser_noEGF_500ng_Report.tsv)
   1000 ng:   30,989 rows  (20260506_113358_nanoPhos_dilser_noEGF_1000ng_Report.tsv)

=== uphos_hela ===
     10 ng:      110 rows  (20260507_105509_uPhos_HeLa_10ng_Report.tsv)
     20 ng:      34

In [31]:
nanoPhos_woEGF   = nanophos_noEGF
nanoPhos_withEGF = nanophos_withEGF
uPhos_HeLa       = uphos_hela


l_nanoPhos_woEGF   = l_nanophos_noEGF
l_nanoPhos_withEGF = l_nanophos_withEGF
l_uPhos_HeLa       = l_uphos_hela

for name, d in [('nanoPhos_woEGF',   nanoPhos_woEGF),
                ('nanoPhos_withEGF', nanoPhos_withEGF),
                ('uPhos_HeLa',       uPhos_HeLa)]:
    print(f"  {name:<20} inputs: {sorted(d)} ng  ({len(d)} datasets)")

  nanoPhos_woEGF       inputs: [10, 20, 50, 100, 200, 500, 1000] ng  (7 datasets)
  nanoPhos_withEGF     inputs: [10, 20, 50, 100, 200, 500, 1000] ng  (7 datasets)
  uPhos_HeLa           inputs: [10, 20, 50, 100, 200, 500, 1000] ng  (7 datasets)


# Supplementary Figure 1A

In [32]:
sel = [75.4, 89.1, 95.9]

In [33]:
labels = ['0mM', '100mM', '200mM']

In [34]:
import importlib, core
importlib.reload(core)
from core import process_ptm_site_report, count_sites_per_sample_ptm_report

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Load + collapse the salt optimization PTM Site Report
salt_df = pd.read_csv(
    'pride_data/analysis_data/revision/figure2/20260518_162116_nanoPhos_optimization_salt_Report.tsv',
    sep='\t'
)
salt = process_ptm_site_report(salt_df, cutoff=0.75)
site_data_salt = salt['site_data']

# 2. Per-sample Class I site counts (defensive filters baked in via the counter)
counts = count_sites_per_sample_ptm_report(salt_df)
n_classI = list(counts.values())
print("Per-sample Class I sites (column order in the report):")
for i, (s, n) in enumerate(counts.items()):
    print(f"  [{i}] {s}  {n:,}")

# 3. Manually supplied values — order matches the column order printed above
labels = ['0mM', '100mM', '200mM']
sel    = [75.4, 89.1, 95.9]

assert len(n_classI) == len(labels), (
    f"Got {len(n_classI)} samples but {len(labels)} labels — if you have "
    f"replicates per condition, you'll need to aggregate before plotting."
)

# 4. Plot — Class I bars + selectivity line on secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Bar(y=n_classI, x=labels, width=0.35,
           marker_line_color='black', marker_color='#642213',
           name='Class I phosphosites'),
    secondary_y=False
)
fig.add_trace(
    go.Scatter(x=labels, y=sel, mode='lines+markers',
               name='Selectivity'),
    secondary_y=True
)
fig.update_yaxes(title_text='Class I phosphosites',
                 secondary_y=False, rangemode='tozero')
fig.update_yaxes(title_text='Selectivity (%)',
                 range=[0, 100], secondary_y=True)
fig.update_traces(marker=dict(size=12, color='darkblue'), secondary_y=True)
fig.update_layout(width=600, height=600, template='simple_white',
                  showlegend=False, xaxis_title='NaCl concentration')
fig.show()
#fig.write_image(r'figures/figure2/suppl_figure1a.pdf', height=600, width=600)


Dropped 4,025 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 26,662 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 13,764 → 13,736.
Final: 13,736 sites × 3 samples.
Per-sample Class I sites (column order in the report):
  [0] 20250411_OA4_Evo11_16p3min_DeOl_nanoPhos_test_0M_salt  8,760
  [1] 20250411_OA4_Evo11_16p3min_DeOl_nanoPhos_test_0p5M_salt  8,704
  [2] 20250411_OA4_Evo11_16p3min_DeOl_nanoPhos_test_1M_salt  7,798


# Supplementary Figure 1B

In [35]:
# Supplementary Figure 1B — phosphopeptide-precursor depth vs protein input (woEGF).
#
# Complementary "identification depth" metric to the Class I sites in Fig. 2b:
#   - A phosphopeptide precursor = a unique phosphorylated precursor (charge-distinct
#     EG.PrecursorId whose EG.ModifiedSequence carries a Phospho modification),
#     localization-INDEPENDENT. This is the Bekker-Jensen et al. 2020 (Nat Commun)
#     definition of "phosphopeptides" ("unique phosphorylated elution group precursors").
#   - Counted per RUN (R.FileName), decoys removed, from the precursor-level reports
#     (the same reports used for the GRAVY panel), so this is MBR-off / single-report and
#     directly comparable to the per-run Class I depth in Fig. 2b (n = 3 per input).
# Plotting convention matches Fig. 2b (boxplot + all individual points).
import os, re
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from core import _hex_to_rgba

PREC_DIR       = r'pride_data/analysis_data/figure2'   # precursor-level reports (as GRAVY panel)
INPUT_NG_ORDER = [10, 20, 50, 100, 200, 500, 1000]

# woEGF dilution-series precursor reports (nanoPhos, unstimulated) -> {input_ng: path}
prec_files = {}
for fn in os.listdir(PREC_DIR):
    if not fn.endswith('Report.tsv') or fn.endswith('_aq.tsv'):
        continue
    if 'nanoPhos_dilser' not in fn or 'withEGF' in fn:
        continue
    if ('woEGF' not in fn) and ('noEGF' not in fn):
        continue
    m = re.search(r'(\d+)ng', fn)
    if m:
        prec_files[int(m.group(1))] = os.path.join(PREC_DIR, fn)

def count_phosphoprecursors_per_run(path):
    """Unique phosphorylated precursors (EG.PrecursorId) per run; decoys removed."""
    df = pd.read_csv(path, sep='\t',
                     usecols=['R.FileName', 'EG.ModifiedSequence', 'EG.PrecursorId', 'EG.IsDecoy'],
                     low_memory=False)
    decoy = (df['EG.IsDecoy'].astype(str).str.lower().isin(['true', '1'])
             if df['EG.IsDecoy'].dtype != bool else df['EG.IsDecoy'])
    df = df[~decoy]
    df = df[df['EG.ModifiedSequence'].astype(str).str.contains('Phospho', case=False, na=False)]
    return df.groupby('R.FileName')['EG.PrecursorId'].nunique().tolist()

woEGF_precursors = {}   # {input_ng: [per-run counts]}
for ng in INPUT_NG_ORDER:
    if ng in prec_files:
        woEGF_precursors[ng] = count_phosphoprecursors_per_run(prec_files[ng])

# summary (same reporting style as Fig. 2b)
print(f"{'input':>7}  {'n':>3}  {'mean':>8}  {'SD':>6}  {'CV%':>5}   per-run")
for ng in sorted(woEGF_precursors):
    c = np.array(woEGF_precursors[ng])
    sd = c.std(ddof=1) if len(c) > 1 else 0.0
    print(f"{ng:>5}ng  {len(c):>3}  {int(round(c.mean())):>8,}  {int(round(sd)):>6,}  "
          f"{100*sd/c.mean():>5.1f}   {list(map(int, c))}")

# Box + all points, same convention as Fig. 2b (distinct colour marks the different metric)
BOX_COLOR   = '#1F5FA6'   # blue = phosphopeptide-precursor depth (vs Class I red in Fig. 2b)
POINT_COLOR = '#393E46'
order = sorted(woEGF_precursors)

fig = go.Figure()
for ng in order:
    ys = woEGF_precursors[ng]
    x  = f"{ng} ng"
    fig.add_trace(go.Box(
        y=ys, x=[x] * len(ys), name=x,
        boxpoints='all', jitter=0.3, pointpos=0,
        marker=dict(size=9, color=POINT_COLOR, line=dict(width=0.5, color='black')),
        line=dict(color=BOX_COLOR, width=1.5),
        fillcolor=_hex_to_rgba(BOX_COLOR, 0.15),
        showlegend=False,
    ))
fig.update_layout(template='plotly_white', width=600, height=600,
                  xaxis_title='Protein input', yaxis_title='Phosphopeptide precursors',
                  showlegend=False)
fig.update_xaxes(categoryorder='array', categoryarray=[f"{ng} ng" for ng in order])
fig.update_yaxes(range=[0, 91000])
fig.show()
#fig.write_image(r'figures/figure2/add_suppl_figure1b.pdf', width=600, height=600)


  input    n      mean      SD    CV%   per-run
   10ng    3     5,086      74    1.4   [5080, 5015, 5162]
   20ng    3    11,386     230    2.0   [11549, 11123, 11485]
   50ng    3    20,147     118    0.6   [20246, 20179, 20016]
  100ng    3    33,075     759    2.3   [33944, 32740, 32542]
  200ng    3    51,114     258    0.5   [50974, 51412, 50956]
  500ng    3    74,203     436    0.6   [73708, 74373, 74528]
 1000ng    3    84,917     146    0.2   [85072, 84783, 84896]


# Supplementary Figure 1C

In [36]:
funscores = pd.read_csv('./data/funscores.csv')

In [37]:
funscores_high = funscores[funscores['probabilities']>= 0.5]

In [38]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

INPUT_NG_ORDER = [10, 20, 50, 100, 200, 500, 1000]

# Per-dilution coverage of high-confidence functional sites.
# Coverage = UNIQUE functional sites detected / total high-confidence functional sites.
# Sites are counted unique by (ProteinId, position) Key — multiplicity collapsed, per
# the project counting policy — so a residue seen on M1 and M2 peptides counts once.
# (Counting rows instead inflates coverage by ~8 points at 1 ug.)
coverage_pct = {}

for ng in INPUT_NG_ORDER:
    if ng not in nanoPhos_woEGF:
        print(f"  skipping {ng} ng (not in nanoPhos_woEGF)")
        continue
    print(f"\n=== {ng} ng ===")
    result = process_ptm_site_report(nanoPhos_woEGF[ng], cutoff=0.75)
    site_data = result['site_data']

    # Key matches funscores 'sites' format: <ProteinId>_<absolute position>
    site_data['Key'] = (site_data['Protein_group'].astype(str)
                        + '_' + site_data['PTM_pos'].astype(str))
    matched   = site_data[site_data['Key'].isin(funscores_high['sites'])]
    n_unique  = matched['Key'].nunique()                 # collapse multiplicity
    pct       = round(100 * n_unique / len(funscores_high), 1)
    coverage_pct[ng] = pct

    print(f"  Unique sites in dataset:           {site_data['Key'].nunique():,}")
    print(f"  Matched functional sites (unique): {n_unique:,}")
    print(f"  Coverage: {pct}%")

# Plot
labels = [f"{ng}ng" for ng in coverage_pct]
y_vals = list(coverage_pct.values())

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=labels, y=y_vals,
    mode='lines+markers',
    marker=dict(size=12, color='darkred',
                line=dict(width=0.5, color='black')),
    line=dict(width=2, color='darkred'),
))
fig.update_layout(
    width=600, height=600,
    template='plotly_white',
    xaxis_title='Protein input',
    yaxis_title='Functional sites coverage (%)',
    showlegend=False,
)
fig.update_yaxes(range=[0, 46])
fig.show()
#fig.write_image(r'figures/figure2/suppl_figure1c.pdf', height=600, width=600)



=== 10 ng ===
Dropped 3,662 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 3,558 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 1,799 → 1,782.
Final: 1,782 sites × 3 samples.
  Unique sites in dataset:           1,744
  Matched functional sites (unique): 667
  Coverage: 6.0%

=== 20 ng ===
Dropped 3,087 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 7,321 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 3,669 → 3,641.
Final: 3,641 sites × 3 samples.
  Unique sites in dataset:           3,559
  Matched functional sites (unique): 1,200
  Coverage: 10.8%

=== 50 ng ===
Dropped 1,864 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 13,143 inten

# Supplementary Figure 1D — functional score vs. phosphosite intensity 

For the largest input (1 µg, noEGF), correlate each Class I site's functional-score probability (Ochoa et al.) against its mean phosphosite intensity. Density (paper style) with a binned-median trend; shown here as Suppl. Fig. 1d.

In [39]:
# Supplementary Figure 1D - functional score vs Class I phosphosite intensity (Reviewer 3, comment 6).
# Points coloured by 2D Gaussian kernel density (viridis). Largest input (1 ug, woEGF).
import numpy as np, pandas as pd, plotly.graph_objects as go
from scipy import stats
from scipy.stats import gaussian_kde
from core import process_ptm_site_report

META = {'Protein_group','Gene_group','PTM_0_aa','PTM_pos','PTM_mult123',
        'PTM_flank','PTM_Collapse_key','PTM_localization','UPD_seq'}

sd = process_ptm_site_report(nanoPhos_woEGF[1000], cutoff=0.75)['site_data']
cols = [c for c in sd.columns if c not in META]
sd['Key']     = sd['Protein_group'].astype(str) + '_' + sd['PTM_pos'].astype(str)
sd['log2int'] = np.log2(np.power(2.0, sd[cols]).mean(axis=1))   # mean linear intensity -> log2

m = (sd.merge(funscores, left_on='Key', right_on='sites', how='inner')
       .dropna(subset=['probabilities', 'log2int']))
rho, p = stats.spearmanr(m['probabilities'], m['log2int'])
r_p, _ = stats.pearsonr(m['probabilities'], m['log2int'])
print(f'matched sites: {len(m):,}  |  Spearman rho = {rho:.3f}  |  Pearson r = {r_p:.3f}')

# OLS correlation line (functional score ~ intensity)
lr = stats.linregress(m['log2int'], m['probabilities'])
xline = np.array([m['log2int'].min(), m['log2int'].max()])
yline = lr.intercept + lr.slope * xline

# 2D Gaussian kernel density for point colouring: KDE fit on a seeded subsample (fast on ~22k
# points), evaluated at every point; points sorted so the densest are drawn on top.
xy = np.vstack([m['log2int'].to_numpy(), m['probabilities'].to_numpy()])
_rng = np.random.default_rng(0)
_fit = _rng.choice(xy.shape[1], size=min(xy.shape[1], 4000), replace=False)
dens = gaussian_kde(xy[:, _fit])(xy)
_order = np.argsort(dens)
xo, yo, do = xy[0][_order], xy[1][_order], dens[_order]

fig = go.Figure()
fig.add_trace(go.Scattergl(
    x=xo, y=yo, mode='markers',
    marker=dict(size=5, color=do, colorscale='Viridis', showscale=True,
                colorbar=dict(title='Point<br>density', thickness=14, len=0.6),
                line=dict(width=0)),
    hoverinfo='skip',
))
fig.add_trace(go.Scatter(
    x=xline, y=yline, mode='lines',
    line=dict(color='red', width=2.5, dash='dash'),
))
fig.update_layout(
    width=640, height=600, template='plotly_white', showlegend=False,
    xaxis_title='Phosphosite intensity (log2)',
    yaxis_title='Functional score (probability)',
)
fig.add_annotation(xref='paper', yref='paper', x=0.04, y=0.96, showarrow=False,
    align='left', font=dict(size=14, color='black'),
    text=f'Spearman ρ = {rho:.2f}<br>Pearson r = {r_p:.2f}<br>n = {len(m):,}')
fig.update_yaxes(range=[-0.02, 1.02])
fig.show()
#fig.write_image(r'figures/figure2/suppl_figure1d.pdf', width=600, height=600)


Dropped 5,001 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 56,854 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 25,988 → 25,905.
Final: 25,905 sites × 3 samples.
matched sites: 22,639  |  Spearman rho = 0.322  |  Pearson r = 0.329


# Supplementary Figure 1E

In [40]:

import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

INPUT_NG_ORDER = [10, 20, 50, 100, 200, 500, 1000]
PREC_DIR = r'pride_data/analysis_data/figure2'

# µPhos phosphopeptide-precursor counts per run (same counter as Suppl. Fig. 1b)
uphos_precursors = {}
for ng in INPUT_NG_ORDER:
    p = os.path.join(PREC_DIR, f'uPhos_HeLa_{ng}ng_Report.tsv')
    if os.path.exists(p):
        uphos_precursors[ng] = count_phosphoprecursors_per_run(p)

summary_rows, ratios, labels = [], [], []
for ng in INPUT_NG_ORDER:
    if ng not in woEGF_precursors or ng not in uphos_precursors:
        continue
    nano = woEGF_precursors[ng]
    uphos_mean = float(np.mean(uphos_precursors[ng]))
    if uphos_mean <= 0:
        continue
    per_rep = [n / uphos_mean for n in nano]          # one ratio per nanoPhos replicate
    ratios += per_rep
    labels += [f'{ng} ng'] * len(per_rep)
    summary_rows.append({'input_ng': ng,
                         'nano_mean': int(np.mean(nano)),
                         'uphos_mean': round(uphos_mean, 1),
                         'ratio_of_means': round(np.mean(nano) / uphos_mean, 1)})

summary = pd.DataFrame(summary_rows).set_index('input_ng')
print(summary.to_string())

df_ratio = pd.DataFrame({'Ratio': ratios, 'ID': labels})
order = [f'{ng} ng' for ng in INPUT_NG_ORDER if ng in woEGF_precursors and ng in uphos_precursors]
fig = px.strip(df_ratio, y='Ratio', x='ID', log_y=True, category_orders={'ID': order})
fig.update_layout(width=600, height=600, template='plotly_white', showlegend=False)
fig.update_traces(marker=dict(size=10, color='#1F5FA6', line=dict(width=0.5, color='black')))
fig.update_yaxes(showgrid=True, gridwidth=0.1, gridcolor='#F3F2F2', griddash='solid')
fig.update_xaxes(showgrid=True, gridwidth=0.1, gridcolor='#F3F2F2', griddash='solid')
fig.add_hline(y=1.0, line=dict(color='black', dash='dot', width=1.5))

# mean ± SD crossbar per condition (n=3) — same as Fig. 2e
stats_d = df_ratio.groupby('ID')['Ratio'].agg(['mean', 'std']).reindex(order)
fig.add_trace(go.Scatter(
    x=stats_d.index, y=stats_d['mean'],
    error_y=dict(type='data', array=stats_d['std'].fillna(0), visible=True,
                 color='black', thickness=1.5, width=10),
    mode='markers',
    marker=dict(symbol='line-ew', size=26, color='black', line=dict(width=1, color='black')),
    showlegend=False, hovertemplate='mean=%{y:.1f}x<extra></extra>',
))
fig.show()
#fig.write_image(r'figures/figure2/suppl_figure1e.pdf', width=600, height=600)


          nano_mean  uphos_mean  ratio_of_means
input_ng                                       
10             5085        65.0            78.2
20            11385       466.7            24.4
50            20147      5093.0             4.0
100           33075     10267.0             3.2
200           51114     14334.3             3.6
500           74203     12318.3             6.0
1000          84917     34551.7             2.5


# Supplementary Figure 1F

In [41]:
import numpy as np
import plotly.graph_objects as go
from core import _hex_to_rgba

INPUT_NG_ORDER = [10, 20, 50, 100, 200, 500, 1000]
META = {'Protein_group', 'Gene_group', 'PTM_0_aa', 'PTM_pos', 'PTM_mult123',
        'PTM_flank', 'PTM_Collapse_key', 'PTM_localization', 'UPD_seq'}

cv_list = []
ng_labels = []
for ng in INPUT_NG_ORDER:
    if ng not in nanoPhos_woEGF:
        continue
    site_data = process_ptm_site_report(nanoPhos_woEGF[ng], cutoff=0.75)['site_data']
    sample_cols = [c for c in site_data.columns if c not in META]

    # CV in linear-intensity space (process_ptm_site_report applies log2 → un-log2 here)
    linear = np.power(2, site_data[sample_cols])
    n_valid = linear.notna().sum(axis=1)
    cv = linear.std(axis=1) / linear.mean(axis=1)
    cv = cv.replace(0, np.nan)
    cv[n_valid < len(sample_cols)] = np.nan

    cv_list.append(cv)
    ng_labels.append(f"{ng}ng")
    print(f"  {ng:>4} ng: n sites with full coverage = {int(cv.notna().sum()):>6,}  "
          f"median CV = {cv.median():.3f}")

cv_flat = np.concatenate([s.values for s in cv_list])
global_median = np.nanmedian(cv_flat)
print(f"\nGlobal median CV: {global_median:.3f}")

# Plot — boxes filled with a translucent version of the point color,
# outline same hue but fully saturated
fig = go.Figure()
for i, (cv, lab) in enumerate(zip(cv_list, ng_labels)):
    c = color_palette[i]
    fig.add_trace(go.Box(
        y=cv, name=lab,
        marker_color=c,                                   # outlier points
        line=dict(color=c, width=1.2),                    # box outline
        fillcolor=_hex_to_rgba(c, 0.35),                  # translucent fill
    ))
fig.update_layout(
    width=900, height=600,
    template='plotly_white',
    showlegend=False,
    xaxis_title='Protein input',
    yaxis_title='Coefficient of variation',
)
fig.add_hline(y=global_median,
              line={'dash': 'dash', 'width': 2, 'color': 'black'},
              annotation_text=f"median CV: {global_median:.3f}",
              annotation_position='top left')
fig.show()
#fig.write_image(r'figures/figure2/suppl_figure1f.pdf', height=600, width=900)


Dropped 3,662 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 3,558 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 1,799 → 1,782.
Final: 1,782 sites × 3 samples.
    10 ng: n sites with full coverage =    674  median CV = 0.189
Dropped 3,087 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 7,321 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 3,669 → 3,641.
Final: 3,641 sites × 3 samples.
    20 ng: n sites with full coverage =  1,422  median CV = 0.185
Dropped 1,864 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 13,143 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 6,418 → 6,390.
Final: 6,390 sites × 3 samples.
    50 ng: n

# Supplementary Figure 1G

In [42]:
# Supplementary Figure 1G — GRAVY hydrophobicity of peptides uniquely identified by
# nanoPhos vs µPhos at 1 µg. This is a PEPTIDE-level comparison, so it uses the
# precursor-level reports + PeptideCollapse(cutoff=0) = ALL identified phosphopeptides
# (matches the original submission). It is intentionally separate from the site-level
# Class I pipeline used for the depth/PCA panels: GRAVY measures whole-peptide
# hydrophobicity (adsorptive recovery), which is upstream of site localization. The
# PTM Site Report only carries a 15-mer flanking window, so it cannot answer this.
import os
from PeptideCollapse_v4 import PeptideCollapse
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from scipy import stats as sst
import numpy as np
import pandas as pd

FASTA    = r'pride_data/analysis_data/figure2/human.fasta'
PREC_DIR = r'pride_data/analysis_data/figure2'   # same precursor reports as v00

def _collapse(fname):
    df = pd.read_csv(os.path.join(PREC_DIR, fname), sep='\t', low_memory=False)
    return PeptideCollapse(verbose=False).process_complete_pipeline(
        df, cutoff=0, fasta_path=FASTA)

nanophos_df = _collapse('nanoPhos_dilser_noEGF_1000ng_Report.tsv')   # 1 µg nanoPhos (noEGF)
uphos_df    = _collapse('uPhos_HeLa_1000ng_Report.tsv')              # 1 µg µPhos

def safe_gravy(seq):
    try:
        return ProteinAnalysis(seq).gravy() if isinstance(seq, str) and seq else np.nan
    except Exception:
        return np.nan

def _seqset(df):
    s = df['UPD_seq'].astype(str).str.replace('*', '', regex=False).str.upper()
    return set(s[s != ''].drop_duplicates())

_nano, _uph   = _seqset(nanophos_df), _seqset(uphos_df)
nanophos_only = _nano - _uph
uphos_only    = _uph  - _nano
shared        = _nano & _uph

def _gravy_array(pool):
    g = np.array([safe_gravy(s) for s in pool])
    return g[~np.isnan(g)]

g_nano_only  = _gravy_array(nanophos_only)
g_shared     = _gravy_array(shared)
g_uphos_only = _gravy_array(uphos_only)

# GRAVY > 0 fractions per pool + Fisher enrichment (nanoPhos-only vs µPhos-only)
print(f"{'pool':<16}{'n':>9}{'G>0':>8}{'G>0 %':>8}{'median':>9}")
for nm, g in [('nanoPhos-only', g_nano_only), ('shared', g_shared), ('uPhos-only', g_uphos_only)]:
    print(f"{nm:<16}{len(g):>9,}{int((g > 0).sum()):>8,}{100*(g > 0).mean():>7.1f}%{np.median(g):>+9.3f}")

_table = np.array([[(g_nano_only > 0).sum(),  (g_uphos_only > 0).sum()],
                   [(g_nano_only <= 0).sum(), (g_uphos_only <= 0).sum()]])
or_, p_one = sst.fisher_exact(_table, alternative='greater')
print(f"\nFisher nanoPhos-only vs uPhos-only: OR = {or_:.2f}, p = {p_one:.2e}")

# Top-10 most hydrophobic peptides unique to nanoPhos (Reviewer 3, comment 123)
print("\nTop 10 hydrophobic nanoPhos-only peptides (GRAVY, sequence):")
for seq, g in sorted(((s, safe_gravy(s)) for s in nanophos_only),
                     key=lambda x: (x[1] if x[1] == x[1] else -99), reverse=True)[:10]:
    print(f"  {g:+.2f}  {seq}")


INFO	PeptideCollapse:PeptideCollapse_v4.py:process_complete_pipeline()- Starting complete pipeline
INFO	PeptideCollapse:PeptideCollapse_v4.py:load_data()- Loaded 271651 rows
INFO	PeptideCollapse:PeptideCollapse_v4.py:load_data()- Total rows loaded: 271651
INFO	PeptideCollapse:PeptideCollapse_v4.py:load_data()- Unique samples (R.FileName): 3
WARNING	PeptideCollapse:PeptideCollapse_v4.py:_check_duplicate_raw_files()- Potential file duplication detected: ['20250721_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dilser_noEGF_1000ng_01', '20250721_OA4_Evo11_16p3min_DeOl_SA_nanoPhos_dilser_noEGF_1000ng_02'] share identical first-100 precursor patterns
INFO	PeptideCollapse:PeptideCollapse_v4.py:load_fasta()- Loaded FASTA with 20597 protein entries
INFO	PeptideCollapse:PeptideCollapse_v4.py:preprocess_data()- Preprocessing: 254751 rows remaining after phospho filter, 16900 non-phospho removed
INFO	PeptideCollapse:PeptideCollapse_v4.py:preprocess_data()- Preprocessing complete: 254751 phospho rows retained

pool                    n     G>0   G>0 %   median
nanoPhos-only      14,208   1,952   13.7%   -0.674
shared              6,408     669   10.4%   -0.768
uPhos-only          2,972     217    7.3%   -0.880

Fisher nanoPhos-only vs uPhos-only: OR = 2.02, p = 1.68e-24

Top 10 hydrophobic nanoPhos-only peptides (GRAVY, sequence):
  +1.78  LLSILLSK
  +1.64  CCSGAIIVLTK
  +1.62  VGIGLVAAASSPLLVK
  +1.56  VLSAMIR
  +1.52  LCAAAASILGK
  +1.50  LVLLTASK
  +1.50  LGSALLIR
  +1.44  AASALLLR
  +1.39  IAIYELLFK
  +1.30  LSTGPALVAAGLAPAEVVVATVASSGVVK


In [43]:
# Supplementary Figure 1G — plot: GRAVY distribution of the three peptide pools.
import plotly.graph_objects as go

fr_nano  = float((g_nano_only  > 0).mean()); n_nano  = len(g_nano_only)
fr_shar  = float((g_shared     > 0).mean()); n_shar  = len(g_shared)
fr_uphos = float((g_uphos_only > 0).mean()); n_uphos = len(g_uphos_only)
med_nano, med_shar, med_uphos = np.median(g_nano_only), np.median(g_shared), np.median(g_uphos_only)

UPHOS_C, NANO_C, SHARE_C = '#dc4726', '#2b2a2a', '#fcf4d7'
OP_BELOW, OP_ABOVE, BIN = 0.5, 0.8, 0.05

all_g   = np.concatenate([g_uphos_only, g_shared, g_nano_only])
edges   = np.arange(np.floor(all_g.min() / BIN) * BIN, np.ceil(all_g.max() / BIN) * BIN + BIN, BIN)
centers = (edges[:-1] + edges[1:]) / 2
opac    = np.where(centers > 0, OP_ABOVE, OP_BELOW)   # dim the GRAVY<0 tail
def _density(g):
    c, _ = np.histogram(g, bins=edges)
    return c / (c.sum() * BIN)

fig = go.Figure()
for d, name, color in [(_density(g_uphos_only), f"µPhos-only (n={n_uphos:,})",   UPHOS_C),
                       (_density(g_shared),      f"shared (n={n_shar:,})",            SHARE_C),
                       (_density(g_nano_only),   f"nanoPhos-only (n={n_nano:,})",     NANO_C)]:
    fig.add_trace(go.Bar(x=centers, y=d, name=name, width=BIN,
                         marker=dict(color=color, opacity=opac,
                                     line=dict(color='white', width=0.3))))
for x, c in [(med_uphos, UPHOS_C), (med_shar, SHARE_C), (med_nano, NANO_C)]:
    fig.add_vline(x=x, line=dict(color=c, dash='dash', width=2))
fig.add_vline(x=0, line=dict(color='black', dash='dot', width=1))

ann = (f"<b>GRAVY &gt; 0 fraction</b><br>µPhos-only: {100*fr_uphos:.1f}%<br>"
       f"shared: {100*fr_shar:.1f}%<br>nanoPhos-only: {100*fr_nano:.1f}%<br><br>"
       f"Fisher (nanoPhos-only vs µPhos-only):<br>OR = {or_:.2f}, p = {p_one:.1e}")
fig.update_layout(barmode='overlay', xaxis_title='GRAVY index',
                  yaxis_title='Probability density', template='plotly_white',
                  width=600, height=600,
                  legend=dict(yanchor='top', y=0.98, xanchor='left', x=0.02,
                              bgcolor='rgba(255,255,255,0.6)'))
fig.show()
# fig.write_image(r'figures/suppl_figure1g.pdf', height=600, width=600)


# Supplementary Figure 1G — hydrophobic nanoPhos-only phosphopeptides
Concrete support for the GRAVY claim: a table of every nanoPhos-only phosphopeptide with GRAVY > 0 (peptide, site, gene/protein, GRAVY), flagging proteins identified **only** via such hydrophobic peptides, plus a GO enrichment of these proteins (expected to be enriched for membrane / integral-membrane components — the sequences most prone to adsorptive loss).

In [44]:
# Hydrophobic nanoPhos-only phosphopeptide table. Reuses nanophos_df, uphos_df,
# nanophos_only and safe_gravy() defined in the Suppl. Fig. 1G cell above.
def _norm(s): return str(s).replace('*', '').upper()

nano_hydro_seqs = set()
for s in nanophos_only:
    g = safe_gravy(s)
    if g == g and g > 0:               # g==g excludes NaN
        nano_hydro_seqs.add(s)

# protein -> all identifying phosphopeptides across BOTH datasets (for the "rescued" flag)
prot_pep = {}
for _df in (nanophos_df, uphos_df):
    for prot, seq in zip(_df['Protein_group'].astype(str), _df['UPD_seq'].astype(str).map(_norm)):
        prot_pep.setdefault(prot, set()).add(seq)
# a protein is "rescued" if ALL its detected phosphopeptides are nanoPhos-only & hydrophobic
rescued_prot = {p for p, peps in prot_pep.items() if peps and peps <= nano_hydro_seqs}

rows = []
for _, r in nanophos_df.iterrows():
    seq = _norm(r['UPD_seq'])
    if seq in nano_hydro_seqs:
        key = str(r['PTM_Collapse_key'])
        rows.append({
            'phosphopeptide': r['UPD_seq'],
            'site': key.split('~')[-1] if '~' in key else key,
            'gene': r['Gene_group'],
            'protein': str(r['Protein_group']),
            'GRAVY': round(safe_gravy(seq), 3),
            'protein_rescued_by_hydrophobic_only': str(r['Protein_group']) in rescued_prot,
        })
hydro_tab = (pd.DataFrame(rows)
             .drop_duplicates(subset=['phosphopeptide', 'site', 'protein'])
             .sort_values('GRAVY', ascending=False).reset_index(drop=True))
print(f'hydrophobic (GRAVY>0) nanoPhos-only phosphosite entries: {len(hydro_tab):,}')
print(f'unique peptides: {hydro_tab["phosphopeptide"].nunique():,} | '
      f'unique proteins: {hydro_tab["protein"].nunique():,}')
print(f'proteins identified ONLY via hydrophobic nanoPhos-only peptides: {len(rescued_prot):,}')
print('\nTop 15 by GRAVY:')
print(hydro_tab.head(15).to_string(index=False))
#hydro_tab.to_csv(r'figures/suppl_table_hydrophobic_nanoPhos_only.csv', index=False)
print('\nsaved: suppl_table_hydrophobic_nanoPhos_only.csv')

hydrophobic (GRAVY>0) nanoPhos-only phosphosite entries: 4,334
unique peptides: 4,262 | unique proteins: 1,409
proteins identified ONLY via hydrophobic nanoPhos-only peptides: 202

Top 15 by GRAVY:
                phosphopeptide            site    gene protein  GRAVY  protein_rescued_by_hydrophobic_only
                     LLs*ILLSK AKAP17A_S440_M1 AKAP17A  Q02040  1.775                                False
                  CCSGAIIVLt*K     PKM_T432_M1     PKM  P14618  1.636                                False
             VGIGLVAAASs*PLLVK    SKIL_S514_M1    SKIL  P12757  1.625                                 True
             VGIGLVAAAs*SPLLVK    SKIL_S513_M1    SKIL  P12757  1.625                                 True
                      VLs*AMIR   DNPH1_S128_M1   DNPH1  O43598  1.557                                False
                  LCAAAAs*ILGK      DDT_S29_M1     DDT  P30046  1.518                                 True
                     LVLLTAs*K  FLYWCH2_S48_M1 FLYWCH

In [45]:
# GO enrichment of the hydrophobic nanoPhos-only proteins (Reviewer 3, comment 5).
# Needs internet (gseapy / Enrichr, HUMAN libraries) -> run in Jupyter.
import gseapy as gp
fg = sorted(set(hydro_tab['gene'].dropna().astype(str)) - {'', 'nan'})
bg = sorted({g for _df in (nanophos_df, uphos_df)
             for g in _df['Gene_group'].dropna().astype(str)} - {'', 'nan'})
print(f'foreground (hydrophobic nanoPhos-only) genes: {len(fg)} | '
      f'background (all detected phospho genes): {len(bg)}')
enr_hydro = gp.enrich(
    gene_list=fg,
    gene_sets=['GO_Cellular_Component_2023', 'GO_Biological_Process_2023',
               'GO_Molecular_Function_2023', 'KEGG_2021_Human'],
    background=bg, outdir=None, cutoff=0.05)
res = enr_hydro.results
print('result columns:', list(res.columns))
sig = res[res['Adjusted P-value'] < 0.05].sort_values('Combined Score', ascending=False)
print(f'significant terms (adj p < 0.05): {len(sig)}')
# select whatever descriptive columns this gseapy version provides
keep = [c for c in ['Gene_set', 'Term', 'Overlap', 'Odds Ratio', 'Adjusted P-value', 'Combined Score', 'Genes']
        if c in sig.columns]
print(sig[keep].head(25).to_string(index=False))
# Expectation: enrichment for membrane / integral-membrane / transmembrane cellular components,
# directly supporting recovery of hydrophobic (membrane-associated) phosphopeptides by nanoPhos.

foreground (hydrophobic nanoPhos-only) genes: 1409 | background (all detected phospho genes): 5204
result columns: ['Gene_set', 'Term', 'P-value', 'Adjusted P-value', 'Old P-value', 'Old adjusted P-value', 'Odds Ratio', 'Combined Score', 'Genes']
significant terms (adj p < 0.05): 6
                  Gene_set                                   Term  Odds Ratio  Adjusted P-value  Combined Score                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [46]:
# GO enrichment of the hydrophobic nanoPhos-only proteins (Reviewer 3, comment 5).
# Needs internet (gseapy / Enrichr, HUMAN libraries) -> run in Jupyter.
import gseapy as gp
fg = sorted(set(hydro_tab['gene'].dropna().astype(str)) - {'', 'nan'})
bg = sorted({g for _df in (nanophos_df, uphos_df)
             for g in _df['Gene_group'].dropna().astype(str)} - {'', 'nan'})
print(f'foreground (hydrophobic nanoPhos-only) genes: {len(fg)} | '
      f'background (all detected phospho genes): {len(bg)}')
enr_hydro = gp.enrich(
    gene_list=fg,
    gene_sets=['GO_Cellular_Component_2023', 'GO_Biological_Process_2023',
               'GO_Molecular_Function_2023', 'KEGG_2021_Human'],
    background=bg, outdir=None, cutoff=0.05)
res = enr_hydro.results
sig = res[res['Adjusted P-value'] < 0.05].sort_values('Combined Score', ascending=False)
go_cols = [c for c in ['Gene_set', 'Term', 'Overlap', 'Odds Ratio', 'P-value',
                       'Adjusted P-value', 'Combined Score', 'Genes'] if c in sig.columns]
print(f'significant GO/KEGG terms (adj p < 0.05): {len(sig)}')
print(sig[go_cols].head(25).to_string(index=False))

# Combined supplementary table: sheet 1 = hydrophobic peptides, sheet 2 = GO enrichment
out = r'figures/suppl_table_hydrophobic_nanoPhos_only.xlsx'
with pd.ExcelWriter(out) as xl:
    hydro_tab.to_excel(xl, sheet_name='hydrophobic_peptides', index=False)
    sig[go_cols].to_excel(xl, sheet_name='GO_enrichment', index=False)
print('saved combined supplementary table (2 sheets):', out)


foreground (hydrophobic nanoPhos-only) genes: 1409 | background (all detected phospho genes): 5204
significant GO/KEGG terms (adj p < 0.05): 6
                  Gene_set                                   Term  Odds Ratio  P-value  Adjusted P-value  Combined Score                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      Genes
GO_Cellular_Component_2023  Microtubule Cytoskeleton (GO:0015630)    

# Supplementary Figures 1H–1N (per-input EGF± PCA)

In [47]:
# Supplementary Figures 1H-1N — data prep (same as Figure 2F: combined +/-EGF series,
# no cross-run normalization, Class I 0.75; global 0.7-completeness filter + imputation).
import importlib, core
importlib.reload(core)
from core import process_ptm_site_report
import re
from collections import defaultdict
import pandas as pd
import plotly.express as px

df_all = pd.read_csv(
    'pride_data/analysis_data/revision/figure2/20260622_091357_nanoPhos_dilser_withEGF_repeat_all_wo_norm_Report.tsv',
    sep='\t'
)
site_data_all = process_ptm_site_report(df_all, cutoff=0.75)['site_data']

PROC_META = {'Protein_group', 'Gene_group', 'PTM_0_aa', 'PTM_pos', 'PTM_mult123',
             'PTM_flank', 'PTM_Collapse_key', 'PTM_localization', 'UPD_seq'}
sample_cols = [c for c in site_data_all.columns if c not in PROC_META]

def parse_egf_ng(name):
    egf = '+' if 'withEGF' in name else '-'
    m = re.search(r'(\d+)ng', name)
    return egf, (int(m.group(1)) if m else None)

condition_to_samples = defaultdict(list)
for s in sample_cols:
    egf, ng = parse_egf_ng(s)
    if ng is not None:
        condition_to_samples[f"EGF{egf}{ng}ng"].append(s)

rename_map = {}
for cond, samples in condition_to_samples.items():
    for i, s in enumerate(sorted(samples)):
        rename_map[s] = f"{cond}{i+1:02d}"
site_data_renamed = site_data_all.rename(columns=rename_map)

AC_META_REQUIRED = {'UPD_seq', 'PTM_localization', 'Protein_group',
                    'Gene_group', 'PTM_Collapse_key'}
extras_to_drop = [c for c in PROC_META
                  if c not in AC_META_REQUIRED and c in site_data_renamed.columns]
site_data_for_ac = site_data_renamed.drop(columns=extras_to_drop)

dict_cond_all = {cond: [rename_map[s] for s in sorted(samples)]
                 for cond, samples in condition_to_samples.items()}

site_data_grouped = ac.set_condition(site_data_for_ac, dict_cond_all)
site_data_filt    = ac.filt_per_percentage(site_data_grouped, 0.7)
site_data_imp     = ac.imputation_normal_distribution(site_data_filt).reset_index()
print(f"site_data_imp: {site_data_imp.shape[0]} samples, {len(condition_to_samples)} conditions")


Dropped 6,577 non-Phospho rows. Mods in input: ['Carbamidomethyl (C)', 'Oxidation (M)', 'Phospho (STY)']. Mods kept: ['Phospho (STY)']
Per-run mask (cutoff=0.75): 0 of 428,019 intensity cells masked (0.0%).
Deduplicated multi-protein rows: 44,673 → 44,568.
Final: 44,568 sites × 42 samples.
site_data_imp: 42 samples, 14 conditions


In [48]:
# Supplementary Figures 1H-1N — per-input EGF+ vs EGF- PCA (one panel per input).
# Subsets the globally filtered+imputed matrix to each input's EGF+/EGF- pair (n=3 each)
# and runs PCA. NOTE: per-input separation is modest (PC variances are low) — describe
# conservatively in the text (Reviewer 1 & 2), not as "clear separation".
INPUT_NG_ORDER = [10, 20, 50, 100, 200, 500, 1000]
EGF_PLUS_C, EGF_MINUS_C = '#C82909', '#0956C8'   # red = EGF+, blue = EGF-

for ng in INPUT_NG_ORDER:
    g_plus, g_minus = f'EGF+{ng}ng', f'EGF-{ng}ng'
    sub = site_data_imp[site_data_imp['group'].isin([g_plus, g_minus])]
    if sub['group'].nunique() < 2:
        print(f"skip {ng} ng (only {sub['group'].nunique()} group present)")
        continue
    pca = ac.run_pca(sub)
    pca_df = pca[0][0]
    fig = px.scatter(pca_df, x='x', y='y', color='group',
                     color_discrete_map={g_plus: EGF_PLUS_C, g_minus: EGF_MINUS_C})
    fig.update_layout(width=600, height=600, template='plotly_white',
                      title=f'{ng} ng', showlegend=False)
    fig.update_traces(marker=dict(size=21, line=dict(width=1, color='black')))
    fig.update_xaxes(title=pca[1]['x_title'])
    fig.update_yaxes(title=pca[1]['y_title'])
    fig.show()
    #fig.write_image(rf'figures/suppl_figure1_pca_{ng}ng.pdf', width=600, height=600)


# Supplementary Figure 1O 

In [49]:
# Supplementary Figure 1O — EGF+ vs EGF- differential per input (AlphaQuant).
# Regeneration block: re-runs the per-input differential and writes results to Z:.
# These results already exist from the original submission, so this is OFF by default
# (set REGENERATE = True to re-run — slow, and overwrites Z:). The differential is
# independent of the Class I site-count reanalysis, so the original AlphaQuant-formatted
# (_aq) reports are reused. The fetch cell below builds the panel from existing results.
import os

AQ_DIR   = r'pride_data/analysis_data/figure2'
RES_BASE = r'pride_data/analysis_data/figure2/aq_analysis/egf'
INPUT_NG_ORDER = [10, 20, 50, 100, 200, 500, 1000]
REGENERATE = False

if REGENERATE:
    for ng in INPUT_NG_ORDER:
        phospho_file = os.path.join(AQ_DIR, f'nanoPhos_dilser_withEGF_repeat_{ng}ng_Report_aq.tsv')
        res_dir = os.path.join(RES_BASE, f'{ng}ng')
        os.makedirs(res_dir, exist_ok=True)
        aq = pd.read_csv(phospho_file, sep='\t')
        labels = list(aq['R.Label'].unique())
        print(f'{ng:>5} ng  R.Label order: {labels}')   # expect 3x EGF+ then 3x EGF-
        smap = pd.DataFrame({'sample': labels,
                             'condition': ['EGF+', 'EGF+', 'EGF+', 'EGF-', 'EGF-', 'EGF-']})
        smap_path = os.path.join(res_dir, 'samplemap_phospho.tsv')
        smap.to_csv(smap_path, sep='\t')
        aq_pipeline.run_pipeline(input_file=phospho_file, samplemap_file=smap_path,
                                 results_dir=res_dir, condpairs_list=[('EGF+', 'EGF-')],
                                 perform_ptm_mapping=True, modification_type='[Phospho (STY)]',
                                 organism='human')
else:
    print('AlphaQuant regeneration OFF (using existing results on Z:). Set REGENERATE = True to rerun.')


AlphaQuant regeneration OFF (using existing results on Z:). Set REGENERATE = True to rerun.


In [50]:
# 1O — fetch differential results + count significant sites per input, then plot.
# (AlphaQuant EGF+ vs EGF- per input; 'green' = significant. Reuses existing results;
#  the regeneration block above is OFF by default.)
RES_BASE = r'pride_data/analysis_data/figure2/aq_analysis/egf'
INPUT_NG_ORDER = [10, 20, 50, 100, 200, 500, 1000]

sig_sites = []
for ng in INPUT_NG_ORDER:
    res = pd.read_csv(os.path.join(RES_BASE, f'{ng}ng', 'EGF+_VS_EGF-.results.tsv'), sep='	')
    sig_sites.append(int((res['color'] == 'green').sum()))   # 'green' = significant in AlphaQuant
print('Significant EGF+ vs EGF- sites per input:',
      dict(zip([f'{ng}ng' for ng in INPUT_NG_ORDER], sig_sites)))

import plotly.graph_objects as go
fig = go.Figure(go.Bar(
    x=[f'{ng}ng' for ng in INPUT_NG_ORDER], y=sig_sites,
    marker_line_color='black', marker_color='#642213',
))
fig.update_layout(width=600, height=600, template='plotly_white', showlegend=False,
                  xaxis_title='Protein input',
                  yaxis_title='Significant phosphosites (EGF+ vs EGF-)')
fig.show()
# fig.write_image(r'figures/suppl_figure1/suppl_figure1o.pdf', height=600, width=600)


Significant EGF+ vs EGF- sites per input: {'10ng': 53, '20ng': 77, '50ng': 257, '100ng': 689, '200ng': 1010, '500ng': 1040, '1000ng': 2679}


In [51]:
# === PRIDE MetaInfo export (run after all panels above) ===
import sys; sys.path.insert(0, r"src")
from metainfo_export import dump_panel
SFIG = 1   # figure number (single source of truth for sheet labels)
from core import count_sites_per_sample_ptm_report, process_ptm_site_report
import numpy as np, pandas as pd
def _try(fn, sheet):
    try: fn()
    except Exception as e: print(f"  [SKIP {sheet}] {type(e).__name__}: {e}")

def _s1a():
    nc=list(counts.values()); labs=["0mM","100mM","200mM"]; selv=[75.4,89.1,95.9]
    dump_panel(pd.DataFrame({"Raw file":list(counts.keys()),"Condition":labs[:len(nc)],
        "Replicate":[1]*len(nc),"Number of class I sites":nc,"Selectivity":selv[:len(nc)]}),f"Suppl Figure {SFIG}a")
_try(_s1a,f"Suppl Figure {SFIG}a")
_try(lambda: dump_panel(pd.DataFrame([{"Condition":f"{ng}ng","Replicate":i+1,"Phosphopeptide precursors":int(v)}
    for ng in sorted(woEGF_precursors) for i,v in enumerate(woEGF_precursors[ng])]),f"Suppl Figure {SFIG}b"),f"Suppl Figure {SFIG}b")
_try(lambda: dump_panel(pd.DataFrame([{"Condition":r["input_ng"],"nanoPhos_precursors_mean":r["nano_mean"],"uPhos_precursors_mean":r["uphos_mean"],"ratio":r["ratio_of_means"]} for r in summary.reset_index().to_dict("records")]),f"Suppl Figure {SFIG}e"),f"Suppl Figure {SFIG}e")
_try(lambda: dump_panel(pd.DataFrame({"Condition":[f"{k}ng" for k in coverage_pct],
    "Percentage":list(coverage_pct.values())}),f"Suppl Figure {SFIG}c"),f"Suppl Figure {SFIG}c")
_try(lambda: dump_panel(pd.concat({lab:cv.reset_index(drop=True) for cv,lab in zip(cv_list,ng_labels)},axis=1),
    f"Suppl Figure {SFIG}f"),f"Suppl Figure {SFIG}f")
def _s1d():
    b=edges; ctr=(b[:-1]+b[1:])/2; w=b[1]-b[0]
    def dens(g):
        c,_=np.histogram(g,bins=b); return c/(c.sum()*w)
    dump_panel(pd.DataFrame({"GRAVY_center":ctr,"uPhos_only":dens(g_uphos_only),
        "shared":dens(g_shared),"nanoPhos_only":dens(g_nano_only)}),f"Suppl Figure {SFIG}g")
_try(_s1d,f"Suppl Figure {SFIG}g")
def _s1ek():
    rows=[]
    for ng in [10,20,50,100,200,500,1000]:
        gp,gm=f"EGF+{ng}ng",f"EGF-{ng}ng"
        sub=site_data_imp[site_data_imp["group"].isin([gp,gm])]
        if sub["group"].nunique()<2: continue
        p=ac.run_pca(sub)[0][0].rename(columns={"x":"PC1","y":"PC2"}).reset_index().rename(columns={"index":"Raw file"})
        p["input"]=f"{ng}ng"; p["sample"]=p["Raw file"]; rows.append(p)
    dump_panel(pd.concat(rows,ignore_index=True),f"Suppl Figure {SFIG}h-n")
_try(_s1ek,f"Suppl Figure {SFIG}h-n")
_try(lambda: dump_panel(pd.DataFrame({"Group":[f"{ng}ng" for ng in INPUT_NG_ORDER],
    "T-test sig phosphosites":sig_sites}),f"Suppl Figure {SFIG}o"),f"Suppl Figure {SFIG}o")
_try(lambda: dump_panel(m.rename(columns={"probabilities":"Functional score","log2int":"Phosphosite intensity (log2)"})[["Functional score","Phosphosite intensity (log2)"]],f"Suppl Figure {SFIG}d"),f"Suppl Figure {SFIG}d")
print(f"Suppl Figure {SFIG} export done.")


  [MetaInfo] wrote 'Suppl Figure 1a'  (3 rows x 5 cols)
  [MetaInfo] wrote 'Suppl Figure 1b'  (21 rows x 3 cols)
  [MetaInfo] wrote 'Suppl Figure 1e'  (7 rows x 4 cols)
  [MetaInfo] wrote 'Suppl Figure 1c'  (7 rows x 2 cols)
  [MetaInfo] wrote 'Suppl Figure 1f'  (25905 rows x 7 cols)
  [MetaInfo] wrote 'Suppl Figure 1g'  (106 rows x 4 cols)
  [MetaInfo] wrote 'Suppl Figure 1h-n'  (42 rows x 6 cols)
  [MetaInfo] wrote 'Suppl Figure 1o'  (7 rows x 2 cols)
  [MetaInfo] wrote 'Suppl Figure 1d'  (22639 rows x 2 cols)
Suppl Figure 1 export done.
